##Library Import##

In [5]:
!pip -q install transformers accelerate -U

import os, json, math, random, time
import numpy as np
import pandas as pd
from typing import List, Dict, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW

from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 116.0 MB/s eta 0:00:00


In [2]:
import json
from google.colab import files
uploaded = files.upload()

Saving CUAD_v1.json to CUAD_v1.json


In [8]:
def find_text(text):
    start = text.find('"')
    end = text.find('"', start + 1)

    if start != -1 and end != -1:
        return text[start+1:end]
    else:
        return None


In [9]:
def extract_label(data):
  for data in contract:
    for paragraphs in contract:
        labels_in_paragraph:{}
        for qas in paragraph:
            return find_text

In [10]:
with open("CUAD_v1.json", "r") as f:
    data = json.load(f)['data']

def extract_labels(data):
    results = []

    for contract in data:
        for paragraph in contract['paragraphs']:
            labels_in_paragraph = []

            for qa in paragraph['qas']:
                if qa['is_impossible'] == False:
                    label = find_text(qa['question'])
                    labels_in_paragraph.append(label)

            results.append({
                "text": paragraph['context'],  # check this field exists
                "labels": labels_in_paragraph
            })

    return results


In [11]:
labelled_paragraphs = extract_labels(data)

In [12]:
for entry in labelled_paragraphs[:3]:
    print(entry['labels'])
    print(entry['text'][:150] + '...')
    print("—" * 50)


['Document Name', 'Parties', 'Agreement Date', 'Effective Date', 'Expiration Date', 'Renewal Term', 'Governing Law', 'Exclusivity', 'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Rofr/Rofo/Rofn', 'Anti-Assignment', 'Price Restrictions', 'Minimum Commitment', 'License Grant', 'Post-Termination Services', 'Warranty Duration', 'Insurance', 'Covenant Not To Sue']
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between El...
——————————————————————————————————————————————————
['Document Name', 'Parties', 'Effective Date', 'Expiration Date', 'Governing Law', 'Change Of Control', 'Anti-Assignment', 'License Grant', 'Audit Rights', 'Uncapped Liability', 'Cap On Liability', 'Warranty Duration']
Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BEEN REQUESTED AS TO CERTAIN PORTIONS OF THIS DOCUMENT. EACH SUCH PORTION,  WHICH HAS BEEN OMITTED HEREIN ...
——————————————————————————————————————————————————
[

In [13]:
def extract_clause_labels	(data):
    results = []

    for contract in data:
        for paragraph in contract['paragraphs']:
            labels_in_paragraph = []

            for qa in paragraph['qas']:
                if qa['is_impossible'] == False:
                    label = qa['id'].split("__")[-1]
                    labels_in_paragraph.append(label)

            results.append({
                "text": paragraph['context'],  # the contract paragraph
                "labels": labels_in_paragraph  # list of labels found
            })

    return results

results = extract_clause_labels(data)
# Print just the first 2 entries
for r in results[:2]:
    print("Context:", r["text"][:100], "...")
    print("Labels:", r["labels"])
    print("-" * 50)

Context: EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREE ...
Labels: ['Document Name', 'Parties', 'Agreement Date', 'Effective Date', 'Expiration Date', 'Renewal Term', 'Governing Law', 'Exclusivity', 'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Rofr/Rofo/Rofn', 'Anti-Assignment', 'Price Restrictions', 'Minimum Commitment', 'License Grant', 'Post-Termination Services', 'Warranty Duration', 'Insurance', 'Covenant Not To Sue']
--------------------------------------------------
Context: Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BEEN REQUESTED AS TO CERTAIN PORTIONS OF THIS DOCUMENT.  ...
Labels: ['Document Name', 'Parties', 'Effective Date', 'Expiration Date', 'Governing Law', 'Change Of Control', 'Anti-Assignment', 'License Grant', 'Audit Rights', 'Uncapped Liability', 'Cap On Liability', 'Warranty Duration']
--------------------------------------------------


In [14]:
from collections import Counter

label_counter = Counter()
for item in results:
    for label in item["labels"]:
        label_counter[label] += 1
print(label_counter)

Counter({'Document Name': 510, 'Parties': 509, 'Agreement Date': 470, 'Governing Law': 437, 'Expiration Date': 413, 'Effective Date': 390, 'Anti-Assignment': 374, 'Cap On Liability': 275, 'License Grant': 255, 'Audit Rights': 214, 'Termination For Convenience': 183, 'Post-Termination Services': 182, 'Exclusivity': 180, 'Renewal Term': 176, 'Insurance': 166, 'Revenue/Profit Sharing': 166, 'Minimum Commitment': 165, 'Non-Transferable License': 138, 'Ip Ownership Assignment': 124, 'Change Of Control': 121, 'Non-Compete': 119, 'Uncapped Liability': 111, 'Notice Period To Terminate Renewal': 111, 'Covenant Not To Sue': 100, 'Rofr/Rofo/Rofn': 85, 'Volume Restriction': 82, 'Competitive Restriction Exception': 76, 'Warranty Duration': 75, 'Irrevocable Or Perpetual License': 70, 'Liquidated Damages': 61, 'No-Solicit Of Employees': 59, 'Affiliate License-Licensee': 59, 'Joint Ip Ownership': 46, 'Non-Disparagement': 38, 'No-Solicit Of Customers': 34, 'Third Party Beneficiary': 32, 'Most Favored N

###Clean CUAD  (doc to clauses, context) map###

In [23]:
import json, itertools
from collections import defaultdict, Counter

In [25]:
def build_doc_clause_map(data):
    clause_types = set()
    clause_frequency = Counter()
    document_clause_map = {}
    co_occurrence = defaultdict(Counter)
    empty_docs = []

    # 1) Scan once to collect clause types and doc-level sets
    for contract in data:
        title = contract["title"]
        paragraphs = contract.get("paragraphs", [])
        clause_set = set()

        for para in paragraphs:
            for qa in para.get("qas", []):
                if qa.get("is_impossible") is False:
                    label = qa["id"].split("__")[-1].strip()
                    clause_types.add(label)
                    clause_set.add(label)
                    clause_frequency[label] += 1

        full_context = " ".join(
            p.get("context", "").strip()
            for p in paragraphs
            if isinstance(p.get("context", ""), str)
        )

        if full_context.strip() == "":
            empty_docs.append(title)

        document_clause_map[title] = {
            "clauses": clause_set,
            "context": full_context
        }

        # co-occurrence (unordered pairs)
        for a, b in itertools.combinations(sorted(clause_set), 2):
            co_occurrence[a][b] += 1
            co_occurrence[b][a] += 1

    # 2) Freeze label space
    clause_labels = sorted(clause_types)
    clause_to_index = {c: i for i, c in enumerate(clause_labels)}

    # 3) Build one-hot label vectors per doc
    for title, info in document_clause_map.items():
        vec = [0] * len(clause_to_index)
        for c in info["clauses"]:
            vec[clause_to_index[c]] = 1
        info["label_vector"] = vec

    return document_clause_map, co_occurrence, empty_docs, clause_to_index, clause_labels, clause_frequency

document_clause_map, co_occurrence, empty_docs, clause_to_index, clause_labels, clause_frequency = build_doc_clause_map(data)
print(f"Labels: {len(clause_labels)}; Empty docs: {len(empty_docs)}")


Labels: 41; Empty docs: 0


###Tokenize for Legal Longformer (4096) + global attention###

In [26]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("lexlms/legal-longformer-large", use_fast=True)
MAX_LEN = 4096


###Pre-tokenize###

In [27]:
input_ids_list, attention_mask_list, labels_list = [], [], []

for title, info in document_clause_map.items():
    ctx = info["context"]
    if not isinstance(ctx, str) or ctx.strip() == "":
        continue

    enc = tokenizer(
        ctx,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors=None,
    )

    input_ids_list.append(enc["input_ids"])
    attention_mask_list.append(enc["attention_mask"])
    labels_list.append(info["label_vector"])

# Global attention: first token global
global_attention_list = []
for attn in attention_mask_list:
    g = [0] * len(attn)
    g[0] = 1
    global_attention_list.append(g)


###Dataset, DataLoaders###

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

In [29]:
class ClauseDataset(Dataset):
    def __init__(self, input_ids, attention_masks, global_masks, labels):
        self.input_ids = input_ids
        self.attn = attention_masks
        self.gmask = global_masks
        self.labels = labels
    def __len__(self): return len(self.input_ids)
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attn[idx], dtype=torch.long),
            "global_attention_mask": torch.tensor(self.gmask[idx], dtype=torch.long),
            "labels_list": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

full_ds = ClauseDataset(input_ids_list, attention_mask_list, global_attention_list, labels_list)

VAL_FRAC = 0.2
train_size = int((1 - VAL_FRAC) * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

BATCH_SIZE = 2  # Longformer is heavy; use 1–2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

###Legal Longformer classifier + training/eval###

In [30]:
import torch.nn as nn
from transformers import AutoModel
from torch.optim import AdamW
import pandas as pd
from sklearn.metrics import classification_report, f1_score

In [31]:
NUM_LABELS = len(clause_labels)

In [32]:
class LegalLongformerClauseClassifier(nn.Module):
    def __init__(self, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("lexlms/legal-longformer-large")
        self.backbone.gradient_checkpointing_enable()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.backbone.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, global_attention_mask=None):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        cls = out.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        return self.classifier(x)

model = LegalLongformerClauseClassifier().to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Some weights of LongformerModel were not initialized from the model checkpoint at lexlms/legal-longformer-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


###pos_weight BCE###

In [33]:
def get_pos_weight(loader, num_labels):
    ys = []
    for b in loader:
        ys.append(b["labels_list"])
    y = torch.cat(ys, dim=0)  # (N, L)
    pos = y.sum(dim=0)
    neg = y.size(0) - pos
    return (neg / (pos + 1e-9)).to(device)

pos_weight = get_pos_weight(train_loader, NUM_LABELS)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optim = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

In [34]:
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    P, Y = [], []
    for b in loader:
        ids  = b["input_ids"].to(device)
        msk  = b["attention_mask"].to(device)
        gmsk = b["global_attention_mask"].to(device)
        y    = b["labels_list"].to(device)
        logits = model(ids, msk, gmsk)
        P.append(torch.sigmoid(logits).cpu())
        Y.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

In [35]:
def evaluate(model, loader, threshold=0.3):
    y_probs, y_true = predict_proba(model, loader)
    y_pred = (y_probs > threshold).astype(int)
    rep = classification_report(y_true, y_pred, target_names=clause_labels, zero_division=0, output_dict=True)
    df = pd.DataFrame(rep).transpose()
    print(f"[thr={threshold}] microF1={df.loc['micro avg','f1-score']:.4f}  macroF1={df.loc['macro avg','f1-score']:.4f}")
    return df, y_probs, y_true

In [38]:
def train(model, train_loader, val_loader, epochs=5, threshold=0.3, patience=2):
    best_f1, best_state, wait = -1.0, None, 0

    # Use new AMP API; only create a scaler on CUDA
    use_cuda_amp = (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda") if use_cuda_amp else None

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0

        for b in train_loader:
            ids  = b["input_ids"].to(device)
            msk  = b["attention_mask"].to(device)
            gmsk = b["global_attention_mask"].to(device)
            y    = b["labels_list"].to(device)

            optim.zero_grad(set_to_none=True)

            if use_cuda_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(ids, msk, gmsk)
                    loss = loss_fn(logits, y)
                scaler.scale(loss).backward()
                scaler.step(optim)
                scaler.update()
            else:
                logits = model(ids, msk, gmsk)
                loss = loss_fn(logits, y)
                loss.backward()
                optim.step()

            total += loss.item()

        print(f"Epoch {ep} train loss: {total / max(1, len(train_loader)):.4f}")

        # eval at fixed threshold
        df_val, _, _ = evaluate(model, val_loader, threshold=threshold)
        micro_f1 = df_val.loc["micro avg", "f1-score"]

        if micro_f1 > best_f1:
            best_f1 = micro_f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    print(f"Best microF1: {best_f1:.4f}")
    return model


In [41]:
pos_weight = get_pos_weight(train_loader, NUM_LABELS)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optim = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


In [42]:
model = train(model, train_loader, val_loader, epochs=5, threshold=0.3, patience=2)

Epoch 1 train loss: 0.9678
[thr=0.3] microF1=0.4709  macroF1=0.4108
Epoch 2 train loss: 0.9569
[thr=0.3] microF1=0.4709  macroF1=0.4108
Epoch 3 train loss: 0.9502
[thr=0.3] microF1=0.4709  macroF1=0.4108
Early stopping.
Best microF1: 0.4709


###Tune thresholds per label (often a free +1–5 points)###

In [43]:
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    P, Y = [], []
    for b in loader:
        ids  = b["input_ids"].to(device)
        msk  = b["attention_mask"].to(device)
        gmsk = b["global_attention_mask"].to(device)
        y    = b["labels_list"].to(device)
        logits = model(ids, msk, gmsk)
        P.append(torch.sigmoid(logits).cpu())
        Y.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def tune_thresholds(y_probs, y_true, labels, grid=np.arange(0.1, 0.91, 0.05)):
    best_t = {}
    for i, lab in enumerate(labels):
        best_f, t_star = 0.0, 0.5
        colp, coly = y_probs[:, i], y_true[:, i]
        for t in grid:
            f = f1_score(coly, (colp > t).astype(int), zero_division=0)
            if f > best_f:
                best_f, t_star = f, t
        best_t[lab] = float(t_star)
    return best_t

# get probs/labels on val
y_probs, y_true = predict_proba(model, val_loader)
best_t = tune_thresholds(y_probs, y_true, clause_labels)

def apply_custom_thresholds(y_probs, labels, tdict):
    y_pred = np.zeros_like(y_probs, dtype=int)
    for i, lab in enumerate(labels):
        y_pred[:, i] = (y_probs[:, i] > tdict.get(lab, 0.5)).astype(int)
    return y_pred

y_pred_custom = apply_custom_thresholds(y_probs, clause_labels, best_t)

from sklearn.metrics import classification_report
rep = classification_report(y_true, y_pred_custom, target_names=clause_labels, zero_division=0, output_dict=True)
print(pd.DataFrame(rep).transpose().loc[["micro avg","macro avg"]])


           precision  recall  f1-score  support
micro avg   0.307987     1.0  0.470932   1288.0
macro avg   0.307987     1.0  0.410835   1288.0


In [46]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.reduction = reduction
    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        pt = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean() if self.reduction == "mean" else loss.sum()

loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
optim = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
model = train(model, train_loader, val_loader, epochs=5, threshold=0.3, patience=2)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 16.12 MiB is free. Process 26518 has 14.72 GiB memory in use. Of the allocated memory 14.48 GiB is allocated by PyTorch, and 106.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)